In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import utils
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shapereader
import matplotlib.ticker as mticker

### Optimal temperatures

In [ ]:
# Map of optimal temperatures (84th percentile of daily maximum temperatures) from ERA5 reanalysis for 1980-2010 period.

wdir = "X:\\user\\liprandicn\\mt-comparison\\honda2014/data/"
ot = xr.open_dataset(wdir+"optimal_temperatures/era5_t2m_max_1980-2010_p84.nc")

fig = plt.figure(figsize=(10,5), dpi=300)
ax = fig.add_subplot(111, projection=ccrs.Robinson(central_longitude=0), frameon=True)
ax.coastlines(resolution='10m', lw = 0.1)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0, color='lightgray', alpha=0.5)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 8}
gl.ylabel_style = {'size': 8}
im = ot.t2m_p84.plot(ax=ax, 
                transform=ccrs.PlateCarree(), 
                cmap='RdBu_r', 
                add_colorbar=False,
                levels=21, 
                vmin=-40, 
                vmax=40)
ax.add_feature(cfeature.OCEAN, facecolor='whitesmoke', zorder=2)

# Manually set colorbar limits
cbar = fig.colorbar(im, ax=ax, orientation='vertical', shrink=0.6, pad=0.05, aspect=20)
# cbar.set_label(r'Daily maximum temperature', fontsize=8)
cbar.ax.set_ylim(-12, 36) 
cbar.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d°C'))

# Remove Antarctica 
land_shp = shapereader.natural_earth(resolution='110m', category='physical', name='land')
land_geoms = shapereader.Reader(land_shp).geometries()
for land in land_geoms:
    if land.bounds[1] < -60:
        ax.add_geometries([land], ccrs.PlateCarree(), 
                          facecolor='whitesmoke', 
                          edgecolor='none', 
                          zorder=2)
        
plt.title('Optimal temperatures (84.5th percentile of daily maximum temperatures for 1980-2010)', fontsize=10)

plt.show()

### Replication Romanello

In [ ]:
wdir = "X:\\user\\liprandicn\\mt-comparison\\honda2014/output/"
temp_type = "heat"
unit = "total"
age_group = "oldest"
region = "World"
years = range(2000,2024)
cause = "All causes"

romanello = utils.LoadMortality(wdir, "replication-romanello-expERF/mortality_replication-romanello_SSP2_ERA5_IMAGE_2000-2023_OT-max_1986-2005_p84", years, region, temp_type, unit, age_group, cause)

fig = plt.figure(figsize=(12,5), dpi=300)
plt.plot(romanello.columns[:-1], romanello.values.flatten()[:-1], label="Romanello et al., 2024", linewidth=2)

formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((5, 5))
plt.gca().yaxis.set_major_formatter(formatter)
plt.grid(False)
plt.legend()
plt.ylim(0, 5e5)
plt.title("Heat related mortality")
plt.ylabel("Excess mortality over 65")
plt.xticks(romanello.columns[:-1], rotation=45)
plt.show()

### Replication Honda

In [ ]:
wdir = "X:\\user\\liprandicn\\mt-comparison\\honda2014/output/"
temp_type = "heat"
unit = "total"
age_group = "oldest"
region = "World"
years = range(2000,2024)
cause = "All causes"

honda = utils.LoadMortality(wdir, "replication-romanello-hondaERF/mortality_replication-romanello_SSP2_ERA5_IMAGE_2000-2023_OT-max_1986-2005_p84", years, region, temp_type, unit, age_group, cause)
plt.plot(honda.columns[:-1], honda.values.flatten()[:-1], label="Honda max", linewidth=2)

formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((5, 5))
plt.gca().yaxis.set_major_formatter(formatter)
plt.grid()
plt.legend()
plt.ylim(0, 5e5)
plt.show()

In [ ]:
wdir = "X:\\user\\liprandicn\\mt-comparison\\honda2014\\data\\risk_function\\"

honda = pd.read_csv(wdir+"interpolated_dataset.csv")
honda_exp = pd.read_csv(wdir+"Interpolated Dataset - Exponential.csv")

plt.plot(honda['daily_temperature'], honda['relative_risk'], label="Honda max", linewidth=2)
plt.plot(honda_exp['daily_temperature'], honda_exp['relative_risk'], label="Honda exp", linewidth=2)

In [ ]:
wdir = "X:\\user\\liprandicn\\mt-comparison\\honda2014\\output\\replication-romanello\\"
mor = pd.read_csv(wdir+"mortality_replication-romanello_SSP2_ERA5_ISO3_2000-2023_OT-max_1986-2005_p84.csv")
EU_COUNTRIES = ["AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "IRL", "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE"]
mor = mor[(mor["units"]=="Total Mortality") &
          (mor["t_type"]=="Heat") &
          (mor["region"].isin(EU_COUNTRIES))][["region", "2013"]]
mor